# H3 — Temperatura e umidade relativa do ar nas aberturas de irrigação

Hipótese: o aumento da temperatura do ar junto à redução da umidade relativa do ar está associado a uma maior necessidade de irrigação. Para manter a comparação com o scatter final de H2, são usados os mesmos eventos e seleções: safra 2024 na linha 4 e safra 2025 na linha 1. Cada ponto representa um evento concluído de irrigação, identificado pela transição `valve_state 0 → 1`; as variáveis ambientais são recuperadas no timestamp da abertura.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    ROOT = (Path.cwd() / '..').resolve()
sys.path.insert(0, str(ROOT))

from src.utils import extrair_eventos_irrigacao

In [2]:
def recupera_ambiente_abertura(df, eventos):
    # Recupera temperatura e umidade relativa do ar no instante da abertura.
    dados = df[['timestamp_10min', 'env_temperature', 'env_humidity']].copy()
    dados['timestamp_10min'] = pd.to_datetime(
        dados['timestamp_10min'], errors='coerce', utc=True
    )
    for coluna in ('env_temperature', 'env_humidity'):
        dados[coluna] = pd.to_numeric(dados[coluna], errors='coerce')
    dados = (
        dados.dropna(subset=['timestamp_10min'])
        .drop_duplicates(subset=['timestamp_10min'])
        .rename(columns={
            'timestamp_10min': 'timestamp_abertura',
            'env_temperature': 'temperatura_ar_evento',
            'env_humidity': 'umidade_relativa_ar_evento',
        })
    )
    return eventos.merge(dados, on='timestamp_abertura', how='left')

In [3]:
# Mesmas seleções do scatter comparativo final do H2.
selecoes_h3 = {
    '2024': [4],
    '2025': [1],
}

tabelas_eventos = []
for safra, linhas in selecoes_h3.items():
    dados_safra = pd.read_csv(
        ROOT / 'data' / 'processed' / f'dataset_m2_{safra}.csv',
        low_memory=False,
    )
    for linha in linhas:
        dados_linha = dados_safra.loc[dados_safra['line'].eq(linha)].copy()
        eventos_linha = extrair_eventos_irrigacao(dados_linha)
        eventos_linha = recupera_ambiente_abertura(
            dados_linha, eventos_linha
        )
        eventos_linha.insert(0, 'line', linha)
        eventos_linha.insert(0, 'safra', safra)
        tabelas_eventos.append(eventos_linha)

tabela_eventos = pd.concat(tabelas_eventos, ignore_index=True)
tabela_eventos = tabela_eventos[[
    'safra', 'line', 'evento', 'timestamp_pre_abertura',
    'timestamp_abertura', 'temperatura_ar_evento',
    'umidade_relativa_ar_evento', 'volume_evento',
]]

In [4]:
# O scatter usa somente eventos com as duas leituras ambientais disponíveis.
dados_scatter = tabela_eventos.dropna(
    subset=['temperatura_ar_evento', 'umidade_relativa_ar_evento']
).copy()

print('Eventos usados no scatter de H3:')
print(
    dados_scatter.groupby('safra').size()
    .rename('amostras')
    .to_string()
)
print(f'\nTotal de eventos usados: {len(dados_scatter)}')
print('\nTabela dos eventos:')
print(tabela_eventos.to_string(index=False))

Eventos usados no scatter de H3:
safra
2024    10
2025     8

Total de eventos usados: 18

Tabela dos eventos:
safra  line  evento    timestamp_pre_abertura        timestamp_abertura  temperatura_ar_evento  umidade_relativa_ar_evento  volume_evento
 2024     4       1 2024-07-26 06:50:00+00:00 2024-07-26 07:00:00+00:00                   31.3                        71.0         1607.0
 2024     4       2 2024-07-27 06:50:00+00:00 2024-07-27 07:00:00+00:00                   31.8                        68.0         1461.0
 2024     4       3 2024-07-29 06:50:00+00:00 2024-07-29 07:00:00+00:00                   32.0                        70.0         1725.0
 2024     4       4 2024-07-30 06:50:00+00:00 2024-07-30 07:00:00+00:00                   32.8                        75.0         1636.0
 2024     4       5 2024-08-02 07:00:00+00:00 2024-08-02 07:10:00+00:00                   32.4                        74.0         1579.0
 2024     4       6 2024-08-03 07:00:00+00:00 2024-08-03 07:1

In [5]:
# Scatter comparativo: temperatura do ar × umidade relativa do ar.
cores_safra = {'2024': '#2171b5', '2025': '#d73027'}

fig, ax = plt.subplots(figsize=(9, 6))
for safra, dados_safra in dados_scatter.groupby('safra', sort=True):
    linhas = ', '.join(
        str(int(linha))
        for linha in sorted(dados_safra['line'].dropna().unique())
    )
    ax.scatter(
        dados_safra['temperatura_ar_evento'],
        dados_safra['umidade_relativa_ar_evento'],
        color=cores_safra.get(safra, '#636363'),
        edgecolors='#252525', linewidths=0.4, alpha=0.85,
        label=f'Ano {safra} — linha(s) {linhas}',
    )

ax.set_title(
    'Temperatura e umidade relativa do ar nas aberturas de irrigação'
)
ax.set_xlabel('Temperatura do ar no instante da abertura (°C)')
ax.set_ylabel('Umidade relativa do ar no instante da abertura (%RH)')
ax.legend(frameon=False)
ax.grid(color='#c6dbef', alpha=0.45)
fig.tight_layout()
saida = ROOT / 'imgs' / 'H3' / 'H3_eventos_temperatura_ar_vs_umidade_relativa_por_safra.png'
saida.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(saida, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Figura salva em: {saida}')

Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H3/H3_eventos_temperatura_ar_vs_umidade_relativa_por_safra.png


In [6]:
# Para o EDA, agora usamos todas as amostras válidas das mesmas linhas.
# Isso permite comparar valve_state = 0 e valve_state = 1.
tabelas_amostras = []
for safra, linhas in selecoes_h3.items():
    dados_safra = pd.read_csv(
        ROOT / 'data' / 'processed' / f'dataset_m2_{safra}.csv',
        low_memory=False,
    )
    dados_safra = dados_safra.loc[dados_safra['line'].isin(linhas)].copy()
    dados_safra['safra'] = safra
    for coluna in ('valve_state', 'env_temperature', 'env_humidity'):
        dados_safra[coluna] = pd.to_numeric(
            dados_safra[coluna], errors='coerce'
        )
    tabelas_amostras.append(dados_safra)

dados_amostras_h3 = pd.concat(tabelas_amostras, ignore_index=True)
dados_amostras_h3 = dados_amostras_h3.loc[
    dados_amostras_h3['valve_state'].isin([0, 1])
].dropna(subset=['env_temperature', 'env_humidity']).copy()
dados_amostras_h3['estado_valvula'] = dados_amostras_h3['valve_state'].astype(int)

print('Amostras válidas por safra e estado da válvula:')
print(
    dados_amostras_h3.groupby(['safra', 'estado_valvula']).size()
    .rename('amostras')
    .to_string()
)

Amostras válidas por safra e estado da válvula:
safra  estado_valvula
2024   0                  9039
       1                   169
2025   0                 16971
       1                    80


In [8]:
# Distribuições exploratórias em uma imagem por safra.
cores_estado = {0: "#cb6d26", 1: "#4dd727"}
rotulos_estado = {0: 'Válvula fechada', 1: 'Válvula aberta'}
variaveis = [
    ('env_temperature', 'Temperatura do ar (°C)', 'Temperatura'),
    ('env_humidity', 'Umidade relativa do ar (%RH)', 'Umidade relativa'),
]

for safra in ('2024', '2025'):
    dados_ano = dados_amostras_h3.loc[
        dados_amostras_h3['safra'].eq(safra)
    ]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, (coluna, xlabel, titulo) in zip(axes, variaveis):
        valores_totais = dados_ano[coluna]
        if coluna == 'env_temperature':
            bins = np.arange(
                np.floor(valores_totais.min()),
                np.ceil(valores_totais.max()) + 1,
                1,
            )
        else:
            bins = np.arange(
                np.floor(valores_totais.min() / 5) * 5,
                np.ceil(valores_totais.max() / 5) * 5 + 5,
                5,
            )

        for estado in (0, 1):
            valores = dados_ano.loc[
                dados_ano['estado_valvula'].eq(estado), coluna
            ]
            ax.hist(
                valores, bins=bins, density=True,
                color=cores_estado[estado], alpha=0.55,
                label=rotulos_estado[estado],
            )
        ax.set_title(titulo)
        ax.set_xlabel(xlabel)
        ax.set_ylabel('Densidade')
        ax.grid(axis='y', color='#c6dbef', alpha=0.45)
        ax.legend(frameon=False)

    fig.suptitle(
        f'Distribuições ambientais por estado da válvula — safra {safra}',
        y=1.03,
    )
    fig.tight_layout()
    saida_distribuicoes = (
        ROOT / 'imgs' / 'H3'
        / f'H3_distribuicoes_temperatura_umidade_por_estado_da_valvula_{safra}.png'
    )
    saida_distribuicoes.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(saida_distribuicoes, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Figura salva em: {saida_distribuicoes}')

Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H3/H3_distribuicoes_temperatura_umidade_por_estado_da_valvula_2024.png
Figura salva em: /mnt/c/Users/ander/Desktop/Faculdade/PI4comIOT/projeto-integrador-iv-mitha/imgs/H3/H3_distribuicoes_temperatura_umidade_por_estado_da_valvula_2025.png
